# Student Performance Prediction System
Upload `student_data.csv` from the ZIP, then run all cells.

In [ ]:
!pip -q install pandas scikit-learn joblib

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select student_data.csv
import pandas as pd
df = pd.read_csv(next(iter(uploaded)))
print("Shape:", df.shape)
display(df.head())
display(df.dtypes)
display(df.isnull().sum())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
import joblib, json

df = df.drop_duplicates().reset_index(drop=True)
y = (df["G3"] < 10).astype(int)
X = df.drop(columns=["G3"])  # Do not use target G3 as an input.
numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()
numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("numeric", numeric_pipe, numeric_cols), ("categorical", categorical_pipe, categorical_cols)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
model = Pipeline([("preprocessor", preprocessor), ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced"))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred, labels=[0,1], target_names=["Not at risk","At risk"], zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, pred, labels=[0,1]))
joblib.dump(model, "student_model.pkl")
X.head(5).to_csv("sample_prediction_input.csv", index=False)
print("Saved student_model.pkl and sample_prediction_input.csv")

## Predict new records
Upload a new CSV containing the same predictor columns (do not include G3).

In [ ]:
new_upload = files.upload()
new_df = pd.read_csv(next(iter(new_upload)))
required = list(model.feature_names_in_)
missing = [c for c in required if c not in new_df.columns]
if missing:
    print("Missing required columns:", missing)
else:
    predictions = model.predict(new_df[required])
    results = new_df.copy()
    results["Prediction"] = ["AT RISK" if int(v)==1 else "NOT AT RISK" for v in predictions]
    display(results.head(20))
    results.to_csv("student_predictions.csv", index=False)
    files.download("student_predictions.csv")